1. Phân tích P.E.A.S
P (Performance Measure): Đưa bàn cờ về đúng trạng thái đích. Hiệu suất được đánh giá bằng số bước di chuyển càng ít càng tốt (Ví dụ: Điểm Performance = 100 - số bước).
E (Environment): Bàn cờ lưới 3x3 bao gồm các số từ 1 đến 8 và một ô trống (biểu diễn bằng số 0). Môi trường có thể quan sát toàn phần (Fully Observable) và mang tính tất định (Deterministic).
A (Actuators): Các cơ cấu chấp hành thực hiện hành động di chuyển ô trống: UP, DOWN, LEFT, RIGHT, STOP.
S (Sensors): Cảm biến nhận diện trạng thái bàn cờ hiện tại, vị trí ô trống, biết được các action hợp lệ, tính toán được khoảng cách heuristic (Manhattan) và biết đã đạt Goal hay chưa.
2. Các thành phần của Agent theo kiến trúc Model-Based
State (Trạng thái hiện tại): Nhận thức (percept) ma trận 3x3 tại thời điểm hiện tại của môi trường.
Model (Mô hình nội tại):

* Hiểu biết về môi trường: Nắm được quy luật dịch chuyển của các ô số khi ô trống thay đổi vị trí.
* Lưu trữ thông tin (Internal State): Lưu trữ trạng thái đích (Goal), tính toán heuristic (khoảng cách Manhattan), lưu trữ số bước đã đi (Visited states), và lưu trữ Plan (Kế hoạch) gồm các bước đi tối ưu chưa được thực thi.
Rules (Tập luật & Lập luận IF):
* IF state hiện tại == goal THEN Action = STOP.
* IF chưa đạt đích VÀ danh sách Plan đang trống THEN Kích hoạt lập luận tạo kế hoạch (dùng A* với Heuristic Manhattan) -> Cập nhật chuỗi hành động vào Plan -> Action = Plan.pop(0).
* IF danh sách Plan đã có sẵn các bước đi THEN Action = Plan.pop(0).

3. Trạng thái môi trường

GOAL STATE (Trạng thái đích)
+------+------+------+
|  1   |  2   |  3   |
+------+------+------+
|  4   |  5   |  6   |
+------+------+------+
|  7   |  8   |      |
+------+------+------+
0 hoặc ô trống = blank tile

In [4]:
import copy
import heapq

# 1. THUẬT TOÁN TÌM ĐƯỜNG A*
class Node:
    def __init__(self, state, parent, action, g, h):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  # Chi phí (số bước)
        self.h = h  # Heuristic
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

def get_blank_pos(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0: return r, c

def get_heuristic(state, goal):
    dist = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                for gr in range(3):
                    for gc in range(3):
                        if goal[gr][gc] == val:
                            dist += abs(r - gr) + abs(c - gc)
    return dist

def get_successors(state):
    successors = []
    r, c = get_blank_pos(state)
    directions = {'UP': (-1, 0), 'DOWN': (1, 0), 'LEFT': (0, -1), 'RIGHT': (0, 1)}
    
    for action, (dr, dc) in directions.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = copy.deepcopy(state)
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            successors.append((new_state, action))
    return successors

def a_star_search(start_state, goal_state):
    start_node = Node(start_state, None, None, 0, get_heuristic(start_state, goal_state))
    open_set = []
    heapq.heappush(open_set, start_node)
    closed_set = set()

    while open_set:
        current = heapq.heappop(open_set)
        state_tuple = tuple(tuple(row) for row in current.state)

        if current.state == goal_state:
            plan = []
            while current.parent:
                plan.append(current.action)
                current = current.parent
            return plan[::-1] 

        closed_set.add(state_tuple)

        for next_state, action in get_successors(current.state):
            next_tuple = tuple(tuple(row) for row in next_state)
            if next_tuple in closed_set: continue
                
            g = current.g + 1
            h = get_heuristic(next_state, goal_state)
            heapq.heappush(open_set, Node(next_state, current, action, g, h))
            
    return []

# 2. ĐỊNH NGHĨA AGENT
class ModelBasedReflexAgent:
    def __init__(self, goal_state):
        self.state = None
        self.action = None
        self.model = {
            'goal': goal_state,
            'plan': [],
            'plan_count': 0
        }

    def agent_function(self, percept):
        self.state = percept
        
        if self.state == self.model['goal']:
            return 'STOP'
            
        if not self.model['plan']:
            self.model['plan'] = a_star_search(self.state, self.model['goal'])
            self.model['plan_count'] += 1
            
        if self.model['plan']:
            self.action = self.model['plan'].pop(0)
        else:
            self.action = 'STOP' 
            
        return self.action

# 3. CHẠY THỬ NGHIỆM
def print_board_with_highlights(state, prev_state=None):
    changed_pos = []
    if prev_state:
        for r in range(3):
            for c in range(3):
                if state[r][c] != prev_state[r][c] and state[r][c] != 0:
                    changed_pos.append((r, c))

    print("+-----+-----+-----+")
    for r in range(3):
        row_str = "|"
        for c in range(3):
            val = state[r][c]
            if val == 0:
                val_str = "     "
            else:
                if (r, c) in changed_pos:
                    val_str = f" [{val}] " 
                else:
                    val_str = f"  {val}  "
            row_str += val_str + "|"
        print(row_str)
        print("+-----+-----+-----+")

goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

# Ma trận mẫu mới để tránh trùng lặp
initial_percept = [
    [1, 2, 3],
    [5, 6, 0],
    [7, 8, 4]
]

agent = ModelBasedReflexAgent(goal_state)
current_percept = copy.deepcopy(initial_percept)
prev_percept = None
step = 0

print("=========================================================")
print(f"TRẠNG THÁI BAN ĐẦU (Step: {step} | Heuristic: {get_heuristic(current_percept, goal_state)})")
print("=========================================================")
print_board_with_highlights(current_percept)

while True:
    step += 1
    action = agent.agent_function(current_percept)
    
    if action == 'STOP':
        print("\n=========================================================")
        print(f"KẾT QUẢ: ĐÃ GIẢI XONG 8 PUZZLE TRONG {step - 1} BƯỚC!")
        print(f"Số lần agent phải lập Plan: {agent.model['plan_count']}")
        print("=========================================================")
        break
        
    prev_percept = copy.deepcopy(current_percept)
    r, c = get_blank_pos(current_percept)
    
    if action == 'UP': current_percept[r][c], current_percept[r-1][c] = current_percept[r-1][c], current_percept[r][c]
    elif action == 'DOWN': current_percept[r][c], current_percept[r+1][c] = current_percept[r+1][c], current_percept[r][c]
    elif action == 'LEFT': current_percept[r][c], current_percept[r][c-1] = current_percept[r][c-1], current_percept[r][c]
    elif action == 'RIGHT': current_percept[r][c], current_percept[r][c+1] = current_percept[r][c+1], current_percept[r][c]
    
    print("\n---------------------------------------------------------")
    print(f"Step: {step:02d} | Action: {action} | Heuristic: {get_heuristic(current_percept, goal_state)}")
    print_board_with_highlights(current_percept, prev_percept)

TRẠNG THÁI BAN ĐẦU (Step: 0 | Heuristic: 5)
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|  5  |  6  |     |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 01 | Action: LEFT | Heuristic: 4
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|  5  |     | [6] |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 02 | Action: LEFT | Heuristic: 3
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|     | [5] |  6  |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 03 | Action: DOWN | Heuristic: 4
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
| [7] |  5  |  6  |
+-----+-----+-----+
|     |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 04 | Action: RIGHT | Heuristic: 5
+--